# Clasificación de Texto con BERT y Hugging Face

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ohtar10/icesi-nlp/blob/main/Sesion4/1-text-classification-with-hf.ipynb)

Autor: Álvaro José Cabrera Lozano, Claudia Lorena Áragon, Josué Cobaleda
Curso: Procesamiento de Lenguaje Natural  
Universidad Icesi


En este notebook implementaremos un clasificador de textos en español sobre cambio climático utilizando transformers pero ya basandonos en un modelo pre-entrenado tipo Bidirectional Encoder Representation from Transformers o BERT por sus siglas y disponible en Hugging Face Hub. El propósito de esta tarea es aprender a utilizar modelos pre-entrenados que por si mismos, sería sumamente costoso entrenar desde cero, tanto por el poder de computo como la disponibilidad de datos de entrenamiento. Entonces gran parte de la labor ya ha sido realizada por nosotros. Nuestra tarea ahora es especializar el modelo en la tarea que tenemos a la mano.

En esta ocasión, vamos a apartarnos de Pytorch Lightning y harémos uso extensivo de la herramientas de Hugging Face, las cuales están especialmente desarrolladas para este tipo de tareas, incluyendo la interacción con modelos pre-entrenados.

#### Referencias
- Dataset: https://huggingface.co/datasets/somosnlp/spa_climate_detection
- [BETO: Spanish BERT](https://huggingface.co/dccuchile/bert-base-spanish-wwm-cased)
- [BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding](http://arxiv.org/abs/1810.04805)
- [Natural Language Processing with Transformers: Building Language Applications With Hugging Face](https://www.amazon.com/Natural-Language-Processing-Transformers-Applications/dp/1098103246)
- [Hugging Face Transformers](https://huggingface.co/docs/transformers/v4.41.3/en/index)
- [Hugging Face Accelerate](https://huggingface.co/docs/accelerate/index)
- [Hugging Face Evaluate](https://huggingface.co/docs/evaluate/v0.4.0/en/index)
- [Hugging Face Datasets](https://huggingface.co/docs/datasets/v2.19.0/en/index)

In [ ]:
import pkg_resources
import warnings

warnings.filterwarnings('ignore')

installed_packages = [package.key for package in pkg_resources.working_set]
IN_COLAB = 'google-colab' in installed_packages

In [ ]:
!test '{IN_COLAB}' = 'True' && !pip install torchinfo evaluate

### Contexto del dataset y carga

Para este mini-proyecto se utilizará el dataset somosnlp/spa_climate_detection, disponible en Hugging Face. Este conjunto de datos en español está diseñado para una tarea de clasificación binaria de texto, cuyo objetivo es identificar si un texto está relacionado con cambio climático o sostenibilidad (1) o si no lo está (0). El dataset contiene 3.680 registros, 2,9 mil ejemplos de entrenamiento y 780 de prueba, y fue construido a partir de la integración de diversas fuentes abiertas, incluyendo textos sobre cambio climático, noticias no relacionadas y publicaciones breves. Además del texto y la etiqueta, incluye variables de contexto como dominio, variedad de español, país de origen y registro lingüístico, lo que permite enriquecer el análisis. Su temática actual, su tamaño manejable y la diversidad de fuentes lo convierten en una opción adecuada para aplicar técnicas de clasificación con BERT y analizar el comportamiento del modelo frente a distintos tipos de textos.

Desde el punto de vista lingüístico, el dataset mezcla principalmente dos variedades de español, es_pe y es_esp, e incluye diferentes registros, desde lenguaje culto hasta más coloquial. También combina textos largos —por ejemplo, párrafos de reportes corporativos— con textos más cortos, como opiniones o publicaciones breves.

 El dataset está disponible en el HuggingFace Hub y puede ser fácilmente descargado con la librería.

In [ ]:
from datasets import load_dataset
import warnings
import os

warnings.filterwarnings("ignore")
dataset = load_dataset('somosnlp/spa_climate_detection', split='train')
dataset.set_format(type="pandas")
df = dataset.to_pandas()
df.head()

In [ ]:
import numpy as np

# Adaptamos las columnas del nuevo dataset al formato esperado por el notebook
if 'question' in df.columns:
    df.rename(columns={'question': 'text'}, inplace=True)
    dataset = dataset.rename_column('question', 'text')

if 'answer' in df.columns:
    df['category'] = df['answer'].map({0: 'No', 1: 'Sí'})
    # Añadimos la columna 'category' al dataset original extrayéndola del dataframe
    if 'category' not in dataset.column_names:
        dataset = dataset.add_column('category', df['category'].tolist())

id2category = dict(enumerate(np.unique(df['category'])))
category2id = {v: k for k, v in id2category.items()}

df['category_id'] = df['category'].apply(lambda x: category2id[x])
df = df[['text', 'category', 'category_id']].sample(frac=1).reset_index(drop=True)
df.head()

En esta ocasión, vamos a explorar un poco más las características del dataset a manera ilustrativa.

Primero, observemos la distribución de las clases:

In [ ]:
import matplotlib.pyplot as plt

df.category.value_counts(ascending=True).plot.barh()
plt.title('Distribución de categorías de clases')
plt.show()

In [ ]:
df.category.value_counts()

En cuanto a la composición de clases, el conjunto de entrenamiento está ligeramente desbalanceado, con aproximadamente 55% de textos positivos y 45% de negativos; en prueba la diferencia es un poco mayor, con 62% positivos y 38% negativos.

Ahora observemos la dispersión de las palabras por cada categoría.

In [ ]:
df['Palabras por Texto'] = df['text'].str.split().apply(len)
df.boxplot('Palabras por Texto', by='category', grid=False, showfliers=False, color='black', figsize=(16, 8))
plt.suptitle('')
plt.xlabel('')
plt.show()

A diferencia del ejercicio anterior donde teníamos 12 categorías con una amplia diversidad, en este nuevo dataset la comparación se reduce a dos clases: textos relacionados ('Sí') y no relacionados ('No') con el cambio climático.

Aquí podemos observar si los textos de una categoría tienden a ser más largos que la otra. Al igual que en el caso anterior, pueden existir *outliers* (textos extremadamente largos o cortos), pero dado que en BERT vamos a manejar cadenas de tamaño fijo durante el entrenamiento, lo que más nos interesa determinar es la media y mediana general de la longitud de las palabras para elegir nuestro `max_length` de forma inteligente.

In [ ]:
df.groupby('category')['Palabras por Texto'].median()

Como podemos observar, la mediana, que como medida de tendencia central tiende a ser más robusta que la media, nos indica que la longitud de los textos para ambas categorías es mucho más corta que en nuestro dataset anterior, rondando entre las $\approx50$ y $\approx75$ palabras.

Esto es muy importante, ya que significa que podríamos utilizar un `max_length` considerablemente menor (por ejemplo, 128 o 256) durante la tokenización para optimizar la memoria y el tiempo de entrenamiento. Aunque si decidimos dejarlo en 512 (el máximo de BERT por defecto) no habrá problema, pues el tokenizador simplemente aplicará padding a los espacios sobrantes.

In [ ]:
dataset.reset_format()

### Definiendo el Tokenizer

Como la idea en este notebook es la de re-utilizar modelos pre-entrenados, algo a tener en cuenta es que para que esto funcione correctamente, debemos **siempre** utilizar el mismo tokenizador que se usó para entrenar el modelo. Recordemos que el tokenizador asigna un código a cada token del vocabulario, y durante la creación de los embeddings, el modelo asume esto como entrada, por lo que su usamos otro tokenizador, el modelo va a ser incapaz de derivar las relaciones semánticas apropiadas.

Para esta tarea, haremos uso de un modelo BERT pre-entrenado en corpus del idioma español. El modelo puede ser encontrado [aquí](https://huggingface.co/dccuchile/bert-base-spanish-wwm-cased), es un modelo entrenado por el [Departamento de Ciencias de la Computación de la Universidad de Chile](https://www.dcc.uchile.cl), el cual según los autores, fue entrenado en un gran corpus de idioma español por lo que resulta un buen candidato para la tarea en cuestión.

In [ ]:
from transformers import AutoTokenizer

model_ckpt = "dccuchile/bert-base-spanish-wwm-cased"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)

Ahora sometamos a prueba el tokenizador con la misma frase del notebook anterior.

In [ ]:
tokenizer.pad_token = '[PAD]'
tokenizer("hola mundo!!", max_length=10, truncation=True, padding='max_length').tokens()

Algo interesante, este tokenizador, por lo menos para las palabras de esta frase de prueba, no separa en tokens distintos estas palabras, esto es justamente la razón por la cual no deberíamos usar un tokenizador diferente con un modelo pre-entrenado, habríamos obtenido tokens diferentes y el modelo no lograría interpretar la semantica como se espera.

Ahora, observemos su vocabulario.

In [ ]:
tokenizer.vocab_size

Tenemos $31002$ tokens, es una cantidad inferior que el modelo manual que intentamos en la lección anterior, pero lo suficientemente amplio para una tarea de NLP.

Ahora, observemos otros parámetros del tokenizador.

In [ ]:
tokenizer.model_max_length

In [ ]:
tokenizer.model_input_names

Primero, observamos que el tokenizador por defecto maneja un tamaño máximo de secuencia de $512$. A diferencia de nuestro caso anterior, la longitud mediana de nuestro nuevo dataset es mucho menor (alrededor de 50-75 palabras), por lo que este tamaño máximo resulta más que suficiente para abarcar casi la totalidad de nuestros textos sin perder información, y el tokenizador simplemente aplicará padding a los espacios sobrantes. Finalmente, observamos sus salidas, las cuales serán las entradas a nuestro modelo. Como ya debemos saber, `input_ids` son los indices de los tokens y `attention_mask` es la máscara de atención cuando tenemos tokens irrelevantes (como el padding) en la cadena.

## Usando un modelo BERT pre-entrenado sencillo

![](https://github.com/Ohtar10/icesi-nlp/blob/main/assets/bert-architecture.png?raw=1)
![](https://github.com/Ohtar10/icesi-nlp/blob/main/assets/bert-tokenization.png?raw=1)
Primero, vamos a probar con un modelo pre-entrenado sin mayor modificación a la capa de clasificación. Hugging Face ofrece una clase utilitaria para inicializar el modelo en modo de clasificación de secuencias, lo cual hará que convenientemente tengamos una capa o cabeza de clasificación en el modelo, justo con la cantidad de clases que definamos.

Recordemos que el modelo y el tokenizador deben pertenecer al mismo "paquete", por lo que invocamos a `from_pretrained` con el mismo id que al tokenizer.

Técnicamente hay dos formas de utilizar un modelo pre-entrenado:

1. Como Featurizer: Es decir, vamos a utilizar todas las capas de codificación del modelo **sin re-entrenarlas** o lo que sería lo mismo **congelandolas** haciendo que lo único que vayamos a entrenar sea el clasificador en si.
2. Fine Tuning: Es decir, dejar todas las capas entrenables y entrenar el modelo en su totalidad, a partir del checkpoint del modelo pre-entrenado, para nuestra tarea especifica.

### BERT pre-entrenado como featurizer (simple)

Una ventaja de este enfoque es que es menos costoso de entrenar, a nivel de recursos de computo, ya que no vamos a calcular gradientes para todas las capas sino para la capa de clasificación. Incluso, no es necesario que utilicemos deep learning para la clasificación final, podemos usar algoritmos clasicos, que parten de los embeddings producidos por el modelo pre-entrenado.

Otra ventaja es que debido a que puede ser mucho más rápido de entrenar que no solo un modelo desde cero, sino más rápido que fine tuning, nuevamente, porque ya no es necesario calcular gradientes para todo el modelo, sino solo para la capa de clasificación de nuestro interes.

In [ ]:
!pip install -q torchinfo evaluate

import torch
from torchinfo import summary
from transformers import AutoModelForSequenceClassification

device = "cuda" if torch.cuda.is_available() else "cpu"
inputs = tokenizer("hola mundo!!!", max_length=10, truncation=True, padding='max_length', return_tensors='pt')

print(f"Input Shapes & Types:")
print({k: (v.shape, v.dtype) for k, v in inputs.items()})

model = AutoModelForSequenceClassification.from_pretrained(model_ckpt, num_labels=len(category2id)).to(device)

# Congelamos los pesos del modelo base para usarlo como featurizer solamente.
for param in model.base_model.parameters():
    param.requires_grad = False


input_sizes = [inputs['input_ids'].shape] * 3
input_types = [inputs['input_ids'].dtype] * 3
with torch.no_grad():
    print(summary(model, input_size=input_sizes, dtypes=input_types, col_names=['input_size', 'output_size', 'num_params', 'trainable']))

Observamos que el modelo tiene una capa `BertModel` que corresponde al modelo pre-entrenado y finaliza con una capa lineal que sería nuestro clasificador, esta es una capa proporcionada para nosotros al momento de inicializar el modelo. Además, observamos que solamente la capa lineal que hemos especificado tiene parámetros entrenables. Entonces, a pesar de que el modelo en si tiene más de 100 millones de parámetros, solamente menos de 10 mil son entrenables.

Observemos todos los modulos registrados en el modelo:

In [ ]:
modules = [m for m, _ in model.named_modules()]
modules

Observamos que la capa final es efectivamente el clasificador. Ahora hagamos una prueba pasando un dummy input:

In [ ]:
with torch.no_grad():
    inputs = {k: v.to(device) for k, v in inputs.items()}
    outputs = model(**inputs)
print({k: v.shape for k, v in outputs.items()})

In [ ]:
outputs

In [ ]:
model.classifier

Observamos que tras invocar el modelo, en efecto, obtenemos una salida de 2 dimensiones, correspondientes al número de clases de nuestro nuevo dataset (Sí y No).

Ahora preparemos los datos para el entrenamiento.

Hugging Face Datasets convenientemente implementa una función para hacer el train-test split en nuestro dataset y automáticamente creará nuevas llaves en el mismo para diferenciarlo.

In [ ]:
training_dataset = dataset.train_test_split(train_size=0.8)
validation_dataset = training_dataset['test'].train_test_split(train_size=0.5)

In [ ]:
from datasets.dataset_dict import DatasetDict

new_dataset = DatasetDict({
    'train': training_dataset['train'],
    'val': validation_dataset['train'],
    'test': validation_dataset['test'],
})
new_dataset

Tenemos nuestros tres conjuntos, sin embargo, esto es la información cruda, debemos preparar los datos para el modelo, lo gual incluye tokenizar y convertir las categorías a ids. Preparamos entonces unas funciones utilitarias.

In [ ]:
def preprocess_function(max_len):
    def _preprocess_function(examples):
        return tokenizer(examples['text'], max_length=max_len, truncation=True, padding='max_length')
    return _preprocess_function

def tokenize(max_len: int = 8):
    def _tokenize(batch):
        return tokenizer(batch['text'], max_length=max_len, truncation=True, padding='max_length')
    return _tokenize

def category_names_2_ids(batch):
    batch['label'] = category2id[batch['category']]
    return batch


Y procedemos a invocarlas. Nótese que para la tokenización, mantendremos la longitud en 512 tokens (el máximo soportado por BERT). Aunque nuestro análisis previo demostró que los textos son mucho más cortos, esto no será problema ya que el tokenizador simplemente aplicará *padding* a los espacios restantes.

In [ ]:
tokenized_dataset = new_dataset.map(preprocess_function(max_len=512), batched=True)
tokenized_dataset = tokenized_dataset.map(category_names_2_ids)
tokenized_dataset

#### Entrenamiento

Ahora procederemos al entrenamiento. Aquí harémos uso de las API de HuggingFace directamente.

In [ ]:
from transformers import Trainer, TrainingArguments
from typing import Dict, Any
import evaluate

# Definimos la función métrica de calidad
accuracy = evaluate.load("accuracy")

def compute_metrics(pred) -> Dict[str, Any]:
    """compute metrics

    Esta función será invocada en
    cada epoca y la utilizaremos para
    calcular la métrica de calidad.
    """
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    # Retorna un diccionario como {'nombre-metrica': valor}
    acc = accuracy.compute(predictions=preds, references=labels)
    return acc


batch_size = 8 if IN_COLAB else 4
logging_steps = len(tokenized_dataset['train']) // batch_size
# Definimos los parámetros globales de entrenamiento
training_args = TrainingArguments(
    output_dir='./hf',
    num_train_epochs=2,
    learning_rate=2e-5,
    per_device_eval_batch_size=batch_size,
    per_device_train_batch_size=batch_size,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    disable_tqdm=False,
    logging_steps=logging_steps,
    report_to='tensorboard'
)

# Y definimos el entrenador, especificando el modelo, datasets y el tokenizador
trainer = Trainer(
    model=model,
    args=training_args,
    compute_metrics=compute_metrics,
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['val'],
    processing_class=tokenizer
)

Con esto nos basta para ejecutar el entrenamiento:

In [ ]:
%%time
trainer.train()

Algo importante a resaltar es que fueron necesarias solo 2 iteraciones para alcanzar una tasa de correctitud de $\approx 70\%$, algo que con el modelo de transformers crudo nos costó muchas más iteraciones. Esto demuestra la importancia de partir de modelos pre-entrenados para este tipo de tareas.

Una ventaja adicional de Hugging Face transformers, es que publica automáticamente el progreso del entrenamiento a tensorboard, en el directorio que hemos especificado. Observemos entonces el proceso de entrenamiento:

In [ ]:
%load_ext tensorboard

In [ ]:
%tensorboard --logdir hf/runs

Y ahora evaluemos el modelo en el conjunto de prueba:

In [ ]:
model.eval()
trainer.evaluate(tokenized_dataset['test'])

Hemos alcanzado una correctitud del $\approx 70\%$.

### Haciendo uso del modelo
Ahora, hagamos predicciónes con el modelo y observemos los resultados.

In [ ]:
predictions = trainer.predict(tokenized_dataset['test'])
predictions

In [ ]:
predicted_labels = np.argmax(predictions.predictions, axis=-1)
test_set = tokenized_dataset['test']
test_set = test_set.add_column('prediction_label', predicted_labels)
test_set = test_set.add_column('prediction', list(map(lambda label: id2category[label], predicted_labels)))
test_set

In [ ]:
columns = ['text', 'label', 'prediction_label', 'category', 'prediction']
test_set.set_format('pandas', columns=columns)
df = test_set.to_pandas()[columns]
df.style.set_table_styles(
    [
        {'selector': 'td', 'props': [('word-wrap', 'break-word')]}
    ]
)
df.head(15)

Los resultados no lucen nada mal, aún se cometen un par de errores, pero de resto, parece bastante aceptable. Observemos los errores.

In [ ]:
errors = df[df['label'] != df['prediction_label']]
errors.head(15)

Los errores parece que son mucho más interesantes que en nuestro mdelo pasado.

## Usando una capa más especializada como clasificador

Quizás podemos hacerlo mejor, hemos observado que por defecto, cuando cargamos la clase, Hugging Face nos da un clasificador muy simple, solo una capa lineal. Pero podemos utilizar un clasificador más complejo que definamos nosotros. Esta técnica seguiría utilizando el resto del modelo como featurizer, pero ahora añadimos complejidad a la capa de clasificación en búsqueda de una mejor calidad en los resutlados.

Entonces, volvemos a cargar el modelo tal como hemos hecho antes:

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(model_ckpt, num_labels=len(category2id)).to(device)

for param in model.base_model.parameters():
    param.requires_grad = False


input_sizes = [inputs['input_ids'].shape] * 3
input_types = [inputs['input_ids'].dtype] * 3
with torch.no_grad():
    print(summary(model, input_size=input_sizes, dtypes=input_types, col_names=['input_size', 'output_size', 'num_params', 'trainable']))

In [ ]:
model.classifier

### Definimos un clasificador propio

Podemos definir cualquier tipo de clasificador que se nos ocurra, siempre que se ajuste a las entradas y salidas del clasificador existente. Vamos a utilizar por ejemplo la misma capa lineal que definimos en el notebook anterior:

In [ ]:
import torch.nn as nn


classifier = nn.Sequential(
    nn.Linear(768, 512),
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(512, 256),
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(256, len(category2id)),
    nn.LogSoftmax(dim=1)
)

# simplemente reemplazamos el clasificador existente por el nuestro:
model.classifier = classifier
with torch.no_grad():
    print(summary(model, input_size=input_sizes, dtypes=input_types, col_names=['input_size', 'output_size', 'num_params', 'trainable']))

Observamos que nuestro modelo ya tiene más parámetros para entrenar, producto de nuestro nuevo clasificador.

Procedemos a definir nuevamente el entrenador.

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    compute_metrics=compute_metrics,
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['val'],
    processing_class=tokenizer
)

Y a entrenar el modelo

In [ ]:
%%time

trainer.train()

Y evaluamos el resultado:

In [ ]:
model.eval()
trainer.evaluate(tokenized_dataset['test'])

Hemos obtenido una mejora considerable en nuestro modelo (pasando de $\approx 70\%$ a $\approx 82\%$), lo cual sugiere que nuestro clasificador más complejo contribuye de manera significativa a una mayor calidad en los resultados.

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(model_ckpt, num_labels=len(category2id)).to(device)
trainer = Trainer(
    model=model,
    args=training_args,
    compute_metrics=compute_metrics,
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['val'],
    processing_class=tokenizer
)

In [ ]:
%%time
trainer.train()

In [ ]:
model.eval()
trainer.evaluate(tokenized_dataset['test'])

Hemos alcanzado una correctitud de $\approx 96\%$ en el conjunto de prueba!

Con fine tuning, todas las capas del modelo ajustarán sus parámetros en respuesta al proceso de entrenamiento, por lo que es natural que la calidad de los resultados se incremente significativamente. Sin embargo, no hay que abusar del fine tuning ya que tiende a hacer overfitting al conjunto con el que se entrena, por eso también es recomendable no entrenarlo demasiado.

# Valor agregado de la tarea
Para analizar mejor el modelo, vamos a calcular más métricas (Classification Report con Precision, Recall y F1-Score) y generar una visualización gráfica de la matriz de confusión usando scikit-learn y seaborn/matplotlib para las predicciones del modelo Fine-Tuned en el conjunto de pruebas. Además, implementaremos una función de inferencia interactiva que reciba un texto crudo, lo procese con el tokenizador, lo pase por el modelo entrenado y aplique Softmax para devolver la clase predicha junto con su probabilidad de confianza.

## Revisión de más Métricas de evaluación y Matriz de Confusión


In [ ]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# 1. Generar predicciones
predictions = trainer.predict(tokenized_dataset['test'])

# 2. Extraer etiquetas predichas
predicted_labels = np.argmax(predictions.predictions, axis=-1)

# 3. Obtener etiquetas reales
true_labels = tokenized_dataset['test']['label']

# 5. Imprimir classification report
print("Classification Report:")
print(classification_report(true_labels, predicted_labels, target_names=['No', 'Sí']))

# 6. Calcular y visualizar matriz de confusión
cm = confusion_matrix(true_labels, predicted_labels)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['No', 'Sí'], yticklabels=['No', 'Sí'])
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix - Fine-Tuned Model')
plt.show()

Los resultados obtenidos en el conjunto de prueba son excepcionales y muy equilibrados. El modelo alcanzó un F1-Score macro y ponderado de 0.97, lo que indica una armonía casi perfecta entre Precision y Recall para ambas clases.

Alta Precisión (0.96 - 0.97): Cuando el modelo clasifica un texto (ya sea como relacionado o no con el cambio climático), acierta en la gran mayoría de los casos, lo que significa que la tasa de falsos positivos es mínima.

Alto Recall (0.96 - 0.97): El modelo es capaz de identificar correctamente casi la totalidad de los textos reales de cada categoría, demostrando que la tasa de falsos negativos también es sumamente baja.

Finalmente, estas métricas confirman que el modelo no presenta sesgos hacia ninguna de las dos clases. A pesar de haber una ligera diferencia en la cantidad de ejemplos (162 de 'Sí' frente a 128 de 'No'), el modelo aprendió a identificar ambas categorías con el mismo nivel de eficacia y confiabilidad.

## Inferencia Interactiva con el Modelo Fine-Tuned

Ahora, vamos a crear una función que permita ingresar cualquier texto en español y obtener no solo la predicción (Sí/No relacionado con el cambio climático), sino también el nivel de confianza (probabilidad) de dicha predicción.

Esta función:
1. Toma un texto crudo.
2. Lo procesa usando el `tokenizer` pre-entrenado.
3. Pasa los tensores generados a través de nuestro modelo Fine-Tuned.
4. Aplica la función `Softmax` a los logits resultantes para convertirlos en probabilidades.
5. Retorna la categoría predicha y su confianza.

In [ ]:
import torch.nn.functional as F

def predict_climate_change_related(text):
    model.eval()
    # 1 & 2. Tomar el texto crudo y procesarlo
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=512).to(device)

    # 3. Pasar los tensores por el modelo
    with torch.no_grad():
        outputs = model(**inputs)

    # 4. Aplicar Softmax a los logits
    logits = outputs.logits
    probabilities = F.softmax(logits, dim=-1)

    # 5. Retornar la categoría predicha y su confianza
    confidence, predicted_class_id = torch.max(probabilities, dim=-1)
    predicted_class = id2category[predicted_class_id.item()]

    return {
        'Text': text,
        'Prediction': predicted_class,
        'Confidence': f"{confidence.item() * 100:.2f}%"
    }

# Prueba de la función con un ejemplo positivo y uno negativo
sample_texts = [
    "Las emisiones de dióxido de carbono han aumentado drásticamente en la última década",
    "El nuevo modelo del iphone 17 a calentado el ambiente tecnológico mundial."
]

for text in sample_texts:
    result = predict_climate_change_related(text)
    print(f"Texto: {result['Text']}")
    print(f"Predicción: {result['Prediction']} | Confianza: {result['Confidence']}\n")


Al probar el modelo con textos inventados, observamos resultados muy reveladores sobre su capacidad de comprensión. En el primer texto, que es directo y factual sobre el clima, el modelo clasifica con una seguridad casi absoluta (99.88%). El segundo texto es el más interesante: utiliza términos que suelen asociarse con el clima como 'calentado' y 'ambiente'. Un modelo simple basado en palabras clave habría fallado aquí, pero nuestro modelo BERT (Fine-Tuned) es capaz de entender el contexto (tecnológico, por la mención del iPhone) y determina correctamente que NO trata sobre cambio climático con un sólido 91.69% de confianza. Esto demuestra el verdadero poder de los Transformers para capturar la semántica real del lenguaje.

## Conclusiones del Caso Práctico (Detección de Cambio Climático)
- En conclusión, el modelo no muestra señales de overfitting, sino una muy buena capacidad de generalización, ya que después del fine-tuning alcanzó una exactitud aproximada del 96.5% sobre el conjunto de prueba, es decir, en datos no vistos durante el entrenamiento. Este resultado se explica, en parte, porque el entrenamiento se limitó a solo 2 épocas, lo que redujo el riesgo de que el modelo memorizara ejemplos específicos o ruido del conjunto de entrenamiento. Además, el buen desempeño en las métricas calculadas sobre el conjunto de prueba confirma que el modelo aprendió patrones relevantes y transferibles, manteniéndose robusto y preciso frente a nuevos datos.
- **El poder de los modelos pre-entrenados:** Trabajar con un modelo base en español (como BETO) demostró ser altamente efectivo para nuestra tarea binaria de detección de textos sobre cambio climático. Vimos una progresión clara: comenzamos con un $\approx 70\%$ de correctitud usando el modelo como un *featurizer* simple, subimos a un $\approx 82\%$ al diseñar un clasificador personalizado más profundo, y finalmente alcanzamos un sobresaliente $\approx 96.5\%$ al realizar *fine tuning* completo. Todo esto en apenas 2 iteraciones de entrenamiento por etapa.
- **Ahorro de recursos y Transfer Learning:** Entrenar un modelo de lenguaje desde cero para entender la semántica del español es sumamente costoso. Aprovechar el *transfer learning* nos permitió obtener resultados de altísima calidad en un dominio específico, ahorrando una cantidad inmensa de tiempo y capacidad de cómputo, adaptándolo rápidamente a nuestras dos categorías (Sí / No).
- **Importancia del idioma y contexto:** Dado que nuestro dataset contenía textos cortos (mediana de 50-75 palabras) en distintas variaciones de español y registros lingüísticos, partir de un modelo fundacional entrenado específicamente en corpus en español fue la clave del éxito. Implementar modelos nuevos desde cero tiene sentido solo cuando la tarea es tan novedosa o específica (por ejemplo, un argot puramente técnico o lenguaje legal muy de nicho) que los modelos actuales no logran capturar el contexto.
- **Balance entre rendimiento y costo computacional:** Aunque el *fine tuning* fue claramente superior en precisión ($\approx 96.5\%$ vs $\approx 82\%$), también evidenciamos que es más costoso computacionalmente (entrenar todos los $\approx 109$ millones de parámetros nos tomó casi 8 minutos, frente a los apenas 3 minutos que tomó entrenar solo los 500 mil parámetros de nuestro clasificador propio). Por ende, si en el futuro nos enfrentamos a un entorno con recursos limitados o sin acceso a GPU, la estrategia de usar el modelo congelado como *featurizer* con una red neuronal pequeña acoplada sigue siendo una alternativa excelente y muy competitiva.